# Roboflow Test set 평가 결과 시각화

이 노트북은 `scripts/evaluate_roboflow_models.py`가 만든 `roboflow-test-eval.json`을 읽어서 혼동행렬과 클래스별 Precision/Recall을 그래프로 보여줍니다. `roboflow-test-eval.md`와 같은 데이터를 시각화한 것입니다.

결과를 새로 뽑으려면 프로젝트 루트에서:

```powershell
backend\.venv\Scripts\python.exe scripts/evaluate_roboflow_models.py --dataset ROBOFLOW_MODEL_ID=backend/model/두번째모델.zip --dataset ROBOFLOW_MODEL_ID_2=backend/model/마지막모델.zip
```

이 노트북을 실행하려면(matplotlib이 필요합니다):

```powershell
uv pip install -r scripts/requirements-analysis.txt --python backend/.venv/Scripts/python.exe
```

그다음 VS Code에서 커널을 `oneul-mwo-damji (.venv)`로 선택하세요.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

# Windows에 기본 포함된 한글 폰트로 설정 (없으면 그래프의 한글 라벨이 네모(□)로 깨짐).
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

DATA_PATH = Path('roboflow-test-eval.json')
with DATA_PATH.open(encoding='utf-8') as f:
    result = json.load(f)

models = result['models']
print(f"{len(models)}개 모델: " + ', '.join(m['model_id'] for m in models))

## 혼동행렬 (모델별)

행: 정답, 열: 예측. 값이 클수록 진한 파란색입니다. 대각선 밖에 값이 있으면 그 두 클래스가 서로 혼동됐다는 뜻입니다.

In [ ]:
def plot_confusion_matrix(model: dict) -> None:
    class_names, labels, matrix = model['class_names'], model['labels'], model['matrix']
    grid = np.array([[matrix[truth][pred] for pred in labels] for truth in class_names])

    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 0.9), max(5, len(class_names) * 0.8)))
    im = ax.imshow(grid, cmap='Blues')
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_yticks(range(len(class_names)))
    ax.set_yticklabels(class_names)
    ax.set_xlabel('예측 (Predicted)')
    ax.set_ylabel('정답 (Truth)')
    ax.set_title(f"혼동행렬 - {model['model_id']}")

    threshold = grid.max() / 2 if grid.max() else 0
    for i in range(len(class_names)):
        for j in range(len(labels)):
            value = grid[i, j]
            ax.text(j, i, str(value), ha='center', va='center', color='white' if value > threshold else 'black')

    fig.colorbar(im, ax=ax, shrink=0.8, label='이미지 수')
    fig.tight_layout()
    plt.show()


for model in models:
    plot_confusion_matrix(model)

## 클래스별 Precision / Recall (모델별)

이 앱은 사용자가 후보를 한 번 더 확인하고 담는 구조라, **Recall보다 Precision(모델이 자신 있게 말한 게 실제로 맞을 확률)이 더 중요합니다.** Precision이 낮은 클래스가 있다면 오검출로 다른 상품이 담길 위험이 큽니다.

In [ ]:
def plot_precision_recall(model: dict) -> None:
    metrics = model['metrics']
    labels = [m['label'] for m in metrics]
    precision = [m['precision'] if m['precision'] is not None else 0 for m in metrics]
    recall = [m['recall'] if m['recall'] is not None else 0 for m in metrics]

    x = np.arange(len(labels))
    width = 0.35
    fig, ax = plt.subplots(figsize=(max(8, len(labels) * 0.9), 5))
    ax.bar(x - width / 2, precision, width, label='Precision')
    ax.bar(x + width / 2, recall, width, label='Recall')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=45, ha='right')
    ax.set_ylim(0, 1.05)
    ax.axhline(1.0, color='gray', linewidth=0.5, linestyle='--')
    ax.set_ylabel('비율')
    ax.set_title(f"클래스별 Precision/Recall - {model['model_id']}")
    ax.legend()
    fig.tight_layout()
    plt.show()


for model in models:
    plot_precision_recall(model)

## 모델 간 Recall 비교

모델이 2개 이상이면, 같은 클래스에 대해 Recall을 나란히 비교합니다. 각 모델은 서로 다른 Test split(zip)으로 평가됐을 수 있다는 점을 감안해서 보세요 — 절대적인 실력 비교라기보다는 참고용입니다.

In [ ]:
if len(models) >= 2:
    base = models[0]
    class_names = base['class_names']
    recalls = {
        model['model_id']: {m['label']: (m['recall'] or 0) for m in model['metrics']}
        for model in models
    }

    x = np.arange(len(class_names))
    width = 0.8 / len(models)
    fig, ax = plt.subplots(figsize=(max(8, len(class_names) * 0.9), 5))
    for i, (model_id, per_class) in enumerate(recalls.items()):
        offset = (i - (len(models) - 1) / 2) * width
        ax.bar(x + offset, [per_class.get(c, 0) for c in class_names], width, label=model_id)
    ax.set_xticks(x)
    ax.set_xticklabels(class_names, rotation=45, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Recall')
    ax.set_title('모델별 Recall 비교')
    ax.legend()
    fig.tight_layout()
    plt.show()
else:
    print('모델이 1개뿐이라 비교 그래프는 생략합니다.')